In [2]:
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [ ]:
import pandas as pd

# ── 1. Load both files ──────────────────────────────────────────────────────
df_results = pd.read_csv('../raw_data/cast_file1.txt', sep='^', dtype=str, encoding='latin-1')
df_entities = pd.read_csv('../raw_data/cast_file2.txt', sep='^', dtype=str, encoding='latin-1')

# ── 2. Clean up column names ────────────────────────────────────────────────
df_results.columns = df_results.columns.str.strip()
df_entities.columns = df_entities.columns.str.strip()

# ── 3. Filter: school-level rows, All Students group, Grade 11 only ─────────
df_results = df_results[
    (df_results['School Code'] != '0000000') &
    (df_results['Student Group ID'] == '1') &
    (df_results['Grade'] == '11')          # ← Grade 11 only
].copy()

# ── 4. Merge in entity info ──────────────────────────────────────────────────
df_entities_clean = df_entities[[
    'County Code', 'District Code', 'School Code',
    'County Name', 'District Name', 'School Name', 'Zip Code'
]].copy()

df = df_results.merge(
    df_entities_clean,
    on=['County Code', 'District Code', 'School Code'],
    how='left'
)

# ── 5. Select and rename columns ─────────────────────────────────────────────
df = df[[
    'County Code', 'District Code', 'School Code',
    'County Name', 'District Name_y', 'School Name_y', 'Zip Code',
    'Total Students Enrolled', 'Total Students Tested',
    'Total Students Tested with Scores', 'Mean Scale Score',
    'Percentage Standard Met and Above', 'Count Standard Met and Above',
    'Percentage Standard Not Met', 'Count Standard Not Met',
]].copy()

df = df.rename(columns={
    'District Name_y': 'District Name',
    'School Name_y': 'School Name',
})

# ── 6. Convert numeric columns ───────────────────────────────────────────────
numeric_cols = [
    'Total Students Enrolled', 'Total Students Tested',
    'Total Students Tested with Scores', 'Mean Scale Score',
    'Percentage Standard Met and Above', 'Count Standard Met and Above',
    'Percentage Standard Not Met', 'Count Standard Not Met',
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── 7. Reset index ───────────────────────────────────────────────────────────
df = df.reset_index(drop=True)

In [12]:
df = df.drop(columns = [col for col in df.columns if col in ['Academic Year', 'Aggregate Level', 'County Name', 'District Name', 'Reporting Category']])
df = df[df['School Code'].isin(school_codes_list)]
df['Students Tested with Scores (Rate)'] = df['Total Students Tested with Scores'] / df['Total Students Enrolled']
df = df.reset_index(drop=True)

In [13]:
df = df[['School Code', 'Percentage Standard Met and Above', 'Students Tested with Scores (Rate)']]

In [16]:
df

,School Code,Percentage Standard Met and Above,Students Tested with Scores (Rate)
0,0106401,70.00,1.000000
1,0130229,67.60,0.997674
2,0130450,70.86,0.936027
3,0132746,2.94,0.971429
4,0134270,71.31,0.990138
...,...,...,...
1151,0101162,27.14,0.952247
1152,5738802,19.92,0.952569
1153,5830013,19.51,1.000000
1154,5835202,21.76,0.946078


In [ ]:
df.to_csv('../cleaned_data/testing_data.csv')